# ST-GCN Joint — NTU60 xsub — 40 effective epochs

This notebook launches the completed baseline directly: 8 outer epochs × `RepeatDataset(times=5)`. It does not repeat smoke tests or batch-size benchmarking. The default is a fresh run; set `RESUME = True` only after a stopped Kaggle session has restored the same work directory.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
CONFIG_PATH = PROJECT_DIR / 'configs/stgcn_ntu60_xsub_40e.py'
WORK_DIR = PROJECT_DIR / 'work_dirs/stgcn_ntu60_xsub_40e'
MODEL_CHECKPOINT_PATH = Path('/kaggle/input/models/duymaingoc/resume/pytorch/default/1/best_acc_top1_epoch_8.pth')
NTU60_URL = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'
RESUME = False  # Fresh 40-effective-epoch run. Change only after interruption.

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('experiment work directory:', WORK_DIR)


In [ ]:
%%bash
set -euo pipefail
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  "importlib-metadata" \
  "mmengine>=0.7.1,<1.0.0" \
  "mmcv-lite==2.1.0"

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  if ! git -C "${MMACTION2_SRC}" rev-parse -q --verify "refs/tags/v1.2.0^{commit}" >/dev/null; then
    git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0
  fi
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q -e "${MMACTION2_SRC}"

# The joint-only experiment does not use ViNLU. Prevent MMAction2 1.2.0 from
# importing its old optional multimodal code against Kaggle's new Transformers.
python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for this skeleton-only environment.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'Could not disable MMAction2 multimodal imports in {path}')
PY


In [ ]:
# Fetch the already-approved NTU60 2D annotations; no repeated inspection.
import glob, os, urllib.request

dst = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
if MODEL_CHECKPOINT_PATH.is_file():
    print('external trained model supplied; dataset download skipped')
elif dst.exists():
    print('dataset ready:', dst)
else:
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if hits:
        os.symlink(os.path.abspath(hits[0]), dst)
        print('linked dataset:', hits[0])
    else:
        temporary = dst.with_suffix(dst.suffix + '.part')
        temporary.unlink(missing_ok=True)
        try:
            urllib.request.urlretrieve(NTU60_URL, temporary)
            temporary.replace(dst)
        except Exception:
            temporary.unlink(missing_ok=True)
            raise
        print('downloaded dataset:', dst)


In [ ]:
# Record the complete resolved configuration before launching training.
# Editable installs are activated only in new Python processes; expose both
# source trees explicitly to this already-running Papermill kernel.
import importlib, sys
for source_dir in (PROJECT_DIR, MMACTION2_DIR):
    source = str(source_dir.resolve())
    if source not in sys.path:
        sys.path.insert(0, source)
importlib.invalidate_caches()
from mmengine.config import Config

LOCAL_FINAL_CHECKPOINT = WORK_DIR / 'epoch_8.pth'
FINAL_CHECKPOINT = (
    MODEL_CHECKPOINT_PATH
    if MODEL_CHECKPOINT_PATH.is_file() else LOCAL_FINAL_CHECKPOINT)
REUSE_TRAINED_RUN = FINAL_CHECKPOINT.is_file()
if not RESUME and (WORK_DIR / 'last_checkpoint').exists() and not REUSE_TRAINED_RUN:
    raise RuntimeError(
        'A prior 40e checkpoint exists. Set RESUME=True to continue it, '
        'or choose a new work directory for a genuinely fresh experiment.')
if REUSE_TRAINED_RUN:
    print('completed training found; will reuse:', FINAL_CHECKPOINT)

resolved_cfg = Config.fromfile(str(CONFIG_PATH))
resolved_cfg.work_dir = str(WORK_DIR)
resolved_path = WORK_DIR / 'resolved_config.py'
resolved_cfg.dump(str(resolved_path))
print('resolved config:', resolved_path)
print('outer epochs:', resolved_cfg.train_cfg.max_epochs)
print('dataset repeats:', resolved_cfg.train_dataloader.dataset.times)
print('effective epochs:', resolved_cfg.train_cfg.max_epochs * resolved_cfg.train_dataloader.dataset.times)
print('batch size / learning rate:', resolved_cfg.train_dataloader.batch_size, '/', resolved_cfg.optim_wrapper.optimizer.lr)


In [ ]:
# Reuse epoch_8.pth after a completed run; otherwise launch/resume training.
import json, os, subprocess, sys, time

def run_training():
    if REUSE_TRAINED_RUN:
        print('training already complete; skipping the 8-epoch subprocess')
        return

    command = [
        sys.executable, str(MMACTION2_DIR / 'tools/train.py'), str(CONFIG_PATH),
        '--work-dir', str(WORK_DIR), '--seed', '42']
    if RESUME:
        command.append('--resume')

    train_env = os.environ.copy()
    train_env['PYTHONPATH'] = (
        str(MMACTION2_DIR) + os.pathsep + str(PROJECT_DIR) + os.pathsep
        + train_env.get('PYTHONPATH', ''))
    train_env['PYTHONUNBUFFERED'] = '1'
    console_path = WORK_DIR / 'training_console.log'
    started = time.time()
    with console_path.open('a', buffering=1) as console:
        process = subprocess.Popen(
            command, cwd=str(PROJECT_DIR), env=train_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            console.write(line)
        return_code = process.wait()
    elapsed = time.time() - started
    (WORK_DIR / 'training_run.json').write_text(json.dumps({
        'command': command, 'resume': RESUME, 'wall_time_sec': elapsed,
        'return_code': return_code
    }, indent=2))
    if return_code != 0:
        raise RuntimeError(f'training failed with return code {return_code}')
    print(f'training completed in {elapsed / 3600:.2f} hours')

run_training()


In [ ]:
# Recovery path: load the completed checkpoint without training again.
trained_model = None
if REUSE_TRAINED_RUN:
    import torch
    from mmengine.runner import load_checkpoint
    from mmaction.registry import MODELS
    from mmaction.utils import register_all_modules

    register_all_modules()
    trained_model = MODELS.build(resolved_cfg.model)
    checkpoint_info = load_checkpoint(
        trained_model, str(FINAL_CHECKPOINT), map_location='cpu')
    model_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trained_model.to(model_device)
    trained_model.eval()
    print('trained ST-GCN loaded:', FINAL_CHECKPOINT, '->', model_device)
    print('checkpoint epoch:', checkpoint_info.get('meta', {}).get('epoch', 8))


In [ ]:
# Produce the required epoch table and final experiment report.
import json

metrics_path = WORK_DIR / 'epoch_metrics.jsonl'
metrics_available = metrics_path.is_file()
if metrics_available:
    raw_records = [
        json.loads(line) for line in metrics_path.read_text().splitlines()
        if line.strip()]
else:
    print('epoch_metrics.jsonl was not uploaded; model loading is unaffected')
    raw_records = [{
        'outer_epoch': epoch, 'effective_epoch': epoch * 5,
        'learning_rate': float('nan'), 'train_loss': float('nan'),
        'val_acc_top1': float('nan'), 'val_acc_top5': float('nan'),
        'gpu_memory_mb': float('nan'), 'epoch_wall_time_sec': float('nan')
    } for epoch in range(1, 9)]
raw_epochs = {record['outer_epoch'] for record in raw_records}
# Runs produced by the original hook were shifted to 2..9. Normalize those
# records in memory; the trained weights and validation values are unchanged.
legacy_epoch_shift = int(1 not in raw_epochs and set(range(2, 10)) <= raw_epochs)
records_by_epoch = {}
for raw_record in raw_records:
    record = dict(raw_record)
    epoch = record['outer_epoch'] - legacy_epoch_shift
    if 1 <= epoch <= 8:
        record['outer_epoch'] = epoch
        record['effective_epoch'] = epoch * 5
        records_by_epoch[epoch] = record
missing_epochs = sorted(set(range(1, 9)) - records_by_epoch.keys())
if missing_epochs:
    raise RuntimeError(f'epoch_metrics.jsonl is incomplete; missing epochs: {missing_epochs}')
records = [records_by_epoch[epoch] for epoch in range(1, 9)]

table_lines = [
    '| Outer Epoch | Effective Epoch | Train Loss | Top-1 | Top-5 |',
    '|-------------|-----------------|------------|-------|-------|']
def metric_text(value, spec):
    return format(value, spec) if metrics_available else 'unavailable'

for record in records:
    table_lines.append(
        f"| {record['outer_epoch']} | {record['effective_epoch']} | "
        f"{metric_text(record['train_loss'], '.6f')} | "
        f"{metric_text(record['val_acc_top1'], '.4f')} | "
        f"{metric_text(record['val_acc_top5'], '.4f')} |")
table = '\n'.join(table_lines)

final = records[-1]
best = (max(records, key=lambda item: item['val_acc_top1'])
        if metrics_available else final)
total_wall = sum(item['epoch_wall_time_sec'] for item in records)
peak_gpu = max(item['gpu_memory_mb'] for item in records)
final_checkpoint = FINAL_CHECKPOINT
if not final_checkpoint.is_file():
    raise FileNotFoundError(f'completed checkpoint not found: {final_checkpoint}')
best_checkpoint = WORK_DIR / f"best_acc_top1_epoch_{best['outer_epoch']}.pth"
if not best_checkpoint.is_file():
    best_checkpoint = FINAL_CHECKPOINT
latest_checkpoint = WORK_DIR / 'latest.pth'
temporary_link = WORK_DIR / '.latest.pth.tmp'
temporary_link.unlink(missing_ok=True)
temporary_link.symlink_to(final_checkpoint.resolve())
temporary_link.replace(latest_checkpoint)
if not metrics_available:
    recovery_issue = (
        'The trained checkpoint loaded successfully, but epoch_metrics.jsonl '
        'was not included with the Kaggle Model; metric fields are unavailable.')
elif legacy_epoch_shift:
    recovery_issue = (
        'Recovered the original hook metadata offset (2..9 -> 1..8); '
        'training weights and metrics were unaffected.')
else:
    recovery_issue = 'None.'

report = f"""EXPERIMENT
----------
Model: ST-GCN
Dataset: NTU60 2D skeleton
Split: Cross-Subject (xsub_train / xsub_val)
Input: Joint, COCO-17
Outer epochs: 8
RepeatDataset: 5
Effective epochs: 40
Batch size: 64

TRAINING
--------
Total training wall time: {metric_text(total_wall, '.1f')} seconds
Final training loss: {metric_text(final['train_loss'], '.6f')}
Peak GPU memory: {metric_text(peak_gpu, '.0f')} MiB
Final learning rate: {metric_text(final['learning_rate'], '.10g')}

BEST VALIDATION
---------------
Best outer epoch: {best['outer_epoch']}
Effective epoch: {best['effective_epoch']}
Top-1: {metric_text(best['val_acc_top1'], '.4f')}
Top-5: {metric_text(best['val_acc_top5'], '.4f')}

FINAL VALIDATION
----------------
Top-1: {metric_text(final['val_acc_top1'], '.4f')}
Top-5: {metric_text(final['val_acc_top5'], '.4f')}

CHECKPOINTS
-----------
Best checkpoint: {best_checkpoint}
Latest checkpoint: {latest_checkpoint}
Final checkpoint: {final_checkpoint}

OUTPUTS
-------
Config: {WORK_DIR / 'resolved_config.py'}
Logs: {WORK_DIR}/<timestamp>/vis_data/scalars.json
Work directory: {WORK_DIR}

ISSUES
------
{recovery_issue}
"""
full_report = table + '\n\n' + report
report_path = PROJECT_DIR / 'artifacts/stgcn_ntu60_xsub_40e_report.txt'
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(full_report)
print(full_report)
print('report:', report_path)
